# Scattering Matrix at a Waveguide Step Discontinuity

This notebook demonstrates how to compute the scattering matrix (S-parameters) at a step discontinuity between two dielectric waveguides using the mode-matching technique.

We consider a silicon nitride (Si₃N₄) channel waveguide that abruptly changes width:
- **Region 1 (z < 0):** 500 nm wide × 300 nm tall
- **Region 2 (z > 0):** 1000 nm wide × 300 nm tall

Both waveguides are clad with silicon dioxide (SiO₂) on all sides. The scattering matrix relates the incident and scattered mode amplitudes at the interface.

Since the structure has two planes of symmetry (x = 0 and y = 0), we compute modes over a quarter of the cross-section using appropriate symmetry boundary conditions.

In [ ]:
# Enable automatic reloading of modules (IPython only)
%load_ext autoreload
%autoreload 2

# Load required modules
import modesolver as mode
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection

In [ ]:
# Material parameters
n_core = 1.99      # Silicon nitride
n_clad = 1.46      # Silicon dioxide

# Waveguide dimensions (nm)
core_height = 300  # Core thickness (same for both waveguides)
w1 = 500           # Waveguide 1 full width
w2 = 1000          # Waveguide 2 full width

# Cladding thickness (nm)
clad_top = 2000    # Top cladding (we simulate upper half only)
clad_side = 2000   # Side cladding

# Grid parameters
dx = 5             # Grid spacing (nm)
dy = 5

# Wavelength
wavelength = 1550  # nm

## Construct Waveguide Meshes

Using quarter-domain symmetry, we only need to model x ≥ 0 and y ≥ 0. The PMC boundary at y = 0 enforces symmetry about the waveguide center, so we only simulate the upper half of the waveguide cross-section.

The `waveguidemesh` function creates a ridge waveguide structure where `rw` is the ridge half-width. Both waveguides must share the same computational domain for mode-matching, so we adjust the side cladding to ensure identical grid dimensions.

In [ ]:
# Layer structure for upper half of waveguide: [half-core, top cladding]
# The bottom of the domain (y=0) is at the waveguide center
n_layers = [n_core, n_clad]
h_layers = [core_height / 2, clad_top]

# Ridge height = half core thickness (entire core layer in upper half)
rh = core_height / 2

# Ridge half-widths
rw1 = w1 / 2       # 250 nm
rw2 = w2 / 2       # 500 nm

# Compute side cladding to give identical domain size
# Total x extent = rw + side, so we need rw1 + side1 = rw2 + side2
side2 = clad_side
side1 = side2 + (rw2 - rw1)  # Extra cladding for narrower waveguide

# Build meshes
x1, y1, xc1, yc1, nx1, ny1, eps1, edges1 = mode.waveguidemesh(
    n_layers, h_layers, rh, rw1, side1, dx, dy, return_edges=True
)
x2, y2, xc2, yc2, nx2, ny2, eps2, edges2 = mode.waveguidemesh(
    n_layers, h_layers, rh, rw2, side2, dx, dy, return_edges=True
)

print(f"Waveguide 1: {ny1} × {nx1} grid, half-width = {rw1} nm")
print(f"Waveguide 2: {ny2} × {nx2} grid, half-width = {rw2} nm")

## Visualize Waveguide Cross-Sections

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.pcolormesh(x1, y1, eps1, cmap='gray')
ax1.add_collection(LineCollection(edges1, colors='white', linewidths=1))
ax1.set_aspect('equal')
ax1.set_xlabel('x (nm)')
ax1.set_ylabel('y (nm)')
ax1.set_title(f'Waveguide 1: w = {w1} nm')

ax2.pcolormesh(x2, y2, eps2, cmap='gray')
ax2.add_collection(LineCollection(edges2, colors='white', linewidths=1))
ax2.set_aspect('equal')
ax2.set_xlabel('x (nm)')
ax2.set_ylabel('y (nm)')
ax2.set_title(f'Waveguide 2: w = {w2} nm')

plt.tight_layout()
plt.show()

## Compute Eigenmodes

We use `wgmodes_yee` with `boundary='0M0E'` to enforce:
- **South (y = 0):** Magnetic wall (PMC) — appropriate for TE mode symmetry
- **East (x = max):** Electric wall (PEC) — appropriate for TE mode symmetry

This gives us the fundamental quasi-TE mode in each waveguide.

In [ ]:
nmodes = 1  # Fundamental mode only

# Compute modes for waveguide 1
modes1 = mode.wgmodes_yee(
    wavelength, n_core, nmodes, dx, dy,
    eps=eps1, boundary='0M0E', collocate=True
)
neff1 = modes1[0]
print(f"Waveguide 1: neff = {neff1:.6f}")

# Compute modes for waveguide 2
modes2 = mode.wgmodes_yee(
    wavelength, n_core, nmodes, dx, dy,
    eps=eps2, boundary='0M0E', collocate=True
)
neff2 = modes2[0]
print(f"Waveguide 2: neff = {neff2:.6f}")

## Visualize Mode Profiles

In [ ]:
# Extract Ex field component (dominant for TE mode)
Ex1 = modes1[1]  # Ex from waveguide 1
Ex2 = modes2[1]  # Ex from waveguide 2

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.pcolormesh(xc1, yc1, np.abs(Ex1), cmap='jet')
ax1.add_collection(LineCollection(edges1, colors='black', linewidths=1))
ax1.set_aspect('equal')
ax1.set_xlabel('x (nm)')
ax1.set_ylabel('y (nm)')
ax1.set_title(f'|Ex| (WG1)\nneff = {neff1:.6f}')

ax2.pcolormesh(xc2, yc2, np.abs(Ex2), cmap='jet')
ax2.add_collection(LineCollection(edges2, colors='black', linewidths=1))
ax2.set_aspect('equal')
ax2.set_xlabel('x (nm)')
ax2.set_ylabel('y (nm)')
ax2.set_title(f'|Ex| (WG2)\nneff = {neff2:.6f}')

plt.tight_layout()
plt.show()

## Compute Scattering Matrix

The `gsm_step` function computes the generalized scattering matrix at the interface using the mode-matching technique. With `normalize=True`, the S-matrix elements directly give power coupling coefficients:

$$
\begin{bmatrix} b_1 \\ b_2 \end{bmatrix} = 
\begin{bmatrix} S_{11} & S_{12} \\ S_{21} & S_{22} \end{bmatrix}
\begin{bmatrix} a_1 \\ a_2 \end{bmatrix}
$$

where:
- $S_{11}$: Reflection coefficient (region 1 → region 1)
- $S_{21}$: Transmission coefficient (region 1 → region 2)
- $S_{12}$: Transmission coefficient (region 2 → region 1)
- $S_{22}$: Reflection coefficient (region 2 → region 2)

In [ ]:
# Compute scattering matrix
S11, S12, S21, S22 = mode.gsm_step(modes1, modes2, dx, dy, normalize=True)

# Extract scalar values (single mode case)
S11 = S11[0, 0]
S12 = S12[0, 0]
S21 = S21[0, 0]
S22 = S22[0, 0]

# Compute power coefficients
R1 = np.abs(S11)**2   # Reflection (from WG1 side)
T12 = np.abs(S21)**2  # Transmission (WG1 → WG2)
R2 = np.abs(S22)**2   # Reflection (from WG2 side)
T21 = np.abs(S12)**2  # Transmission (WG2 → WG1)

print("Scattering Matrix (amplitude coefficients):")
print(f"  S11 = {S11: .6f}  (reflection, WG1 side)")
print(f"  S21 = {S21: .6f}  (transmission, WG1 → WG2)")
print(f"  S12 = {S12: .6f}  (transmission, WG2 → WG1)")
print(f"  S22 = {S22: .6f}  (reflection, WG2 side)")
print()
print("Power Coefficients:")
print(f"  Reflection (WG1 side):   R = |S11|² = {R1:.4f} ({R1*100:.2f}%)")
print(f"  Transmission (WG1→WG2):  T = |S21|² = {T12:.4f} ({T12*100:.2f}%)")
print(f"  Reflection (WG2 side):   R = |S22|² = {R2:.4f} ({R2*100:.2f}%)")
print(f"  Transmission (WG2→WG1):  T = |S12|² = {T21:.4f} ({T21*100:.2f}%)")
print()
print(f"Power conservation check (WG1 side): R + T = {R1 + T12:.6f}")
print(f"Power conservation check (WG2 side): R + T = {R2 + T21:.6f}")

## Summary

The scattering matrix reveals how light couples at the abrupt width transition:

- **Transmission** is high because both waveguides support well-confined fundamental modes with significant spatial overlap.
- **Reflection** is small but nonzero due to the mode mismatch at the interface.
- **Power conservation** (R + T = 1) is satisfied, confirming the validity of the mode-matching calculation.

The asymmetry between forward (narrow → wide) and backward (wide → narrow) transmission arises from the different mode profiles in each waveguide.